In [ ]:
import json
import random
import re
from collections import defaultdict

with open('C:/roampal-labs/data/locomo_full.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

exam = data['locomo_exam']
memories = data['memories']

print('Total exam entries:', len(exam))

# Group by category
by_cat = defaultdict(list)
for i, e in enumerate(exam):
    by_cat[e['category_name']].append((i, e))

print('Categories:', {k: len(v) for k, v in by_cat.items()})

# Sample 10 per category with seed=42
SEED = 42
CATS = ['commonsense', 'adversarial', 'temporal', 'single-hop', 'multi-hop']
N_PER_CAT = 10

rng = random.Random(SEED)
sampled = []
for cat in CATS:
    pool = by_cat[cat]
    chosen = rng.sample(pool, N_PER_CAT)
    sampled.extend(chosen)

print(f'Total sampled: {len(sampled)}')
for orig_idx, e in sampled:
    print(f"[{e['category_name']}] conv={e['conv_idx']} | Q: {e['question'][:70]} | GT: {e['ground_truth'][:60]}")

# Build lookup: conv_idx -> list of memory chunks
conv_chunks = defaultdict(list)
for m in memories:
    conv_chunks[m['conv_idx']].append(m['content'])

print('\nConversation chunk counts:')
for c in sorted(conv_chunks.keys()):
    print(f'  conv {c}: {len(conv_chunks[c])} chunks')


In [ ]:
def search_chunks(conv_idx, keywords, top_n=5):
    """Find chunks containing any of the keywords (case-insensitive)."""
    chunks = conv_chunks[conv_idx]
    hits = []
    for chunk in chunks:
        score = sum(1 for kw in keywords if kw.lower() in chunk.lower())
        if score > 0:
            hits.append((score, chunk))
    hits.sort(key=lambda x: -x[0])
    return hits[:top_n]

def extract_keywords(question, ground_truth):
    """Extract meaningful keywords from Q + GT."""
    text = (question + ' ' + ground_truth).lower()
    stop = {'the', 'a', 'an', 'is', 'was', 'did', 'has', 'have', 'had', 'do',
            'does', 'what', 'when', 'where', 'who', 'how', 'why', 'which',
            'and', 'or', 'but', 'in', 'on', 'at', 'to', 'of', 'for', 'with',
            'not', 'no', 'that', 'this', 'are', 'be', 'been', 'her', 'his',
            'she', 'he', 'it', 'they', 'their', 'there', 'about', 'would',
            'could', 'should', 'will', 'from', 'by', 'as', 'if', 'so'}
    words = re.findall(r'[a-z]+', text)
    return [w for w in words if w not in stop and len(w) > 2]

# Full verification run - output all 50 with evidence
results_detail = []
for orig_idx, e in sampled:
    kw = extract_keywords(e['question'], e['ground_truth'])
    hits = search_chunks(e['conv_idx'], kw, top_n=5)
    results_detail.append({
        'orig_idx': orig_idx,
        'question': e['question'],
        'ground_truth': e['ground_truth'],
        'category': e['category_name'],
        'conv_idx': e['conv_idx'],
        'keywords': kw[:10],
        'num_hits': len(hits),
        'best_chunks': [h[1][:500] for h in hits[:3]]
    })

# Save intermediate for manual review
with open('C:/roampal-labs/results/gt_verification_intermediate.json', 'w') as f:
    json.dump(results_detail, f, indent=2)
print('Saved intermediate results')
print(f'Entries with hits: {sum(1 for r in results_detail if r["num_hits"] > 0)}')
print(f'Entries with NO hits: {sum(1 for r in results_detail if r["num_hits"] == 0)}')
for r in results_detail:
    print(f"\n[{r['category']}] conv={r['conv_idx']} hits={r['num_hits']}")
    print(f"  Q: {r['question']}")
    print(f"  GT: {r['ground_truth']}")
    if r['best_chunks']:
        print(f"  BEST: {r['best_chunks'][0][:200]}")
